In [3]:
#Step 1: importing all the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score,precision_score,f1_score,roc_auc_score,classification_report



In [4]:
# Step 2.1: Loading the data
filepath_train="C:\\Users\\MY PC\\OneDrive\\Documents\\Desktop\\Research\\train.csv"
filepath_test="C:\\Users\\MY PC\\OneDrive\\Documents\\Desktop\\Research\\test.csv"
train=pd.read_csv(filepath_train)

# Step 2.2: exploring the given data
print("Rows, Columns=",train.shape)
train.head(3)

Rows, Columns= (200000, 202)


,ID_code,target,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
0,train_0,0,8.9255,-6.7863,11.9081,5.0930,11.4607,-9.2834,5.1187,18.6266,...,4.4354,3.9642,3.1364,1.6910,18.5227,-2.3978,7.8784,8.5635,12.7803,-1.0914
1,train_1,0,11.5006,-4.1473,13.8588,5.3890,12.3622,7.0433,5.6208,16.5338,...,7.6421,7.7214,2.5837,10.9516,15.4305,2.0339,8.1267,8.7889,18.3560,1.9518
2,train_2,0,8.6093,-2.7457,12.0805,7.8928,10.5825,-9.0837,6.9427,14.6155,...,2.9057,9.7905,1.6704,1.6858,21.6042,3.1417,-6.5213,8.2675,14.7222,0.3965


In [5]:
# Step 2.3: checking for missing values
print("Missing values: ",train.isnull().sum().sum())
print('------------------------')
print(train['target'].value_counts())
print('------------------------')
print(train['target'].value_counts(normalize=True)) 

Missing values:  0
------------------------
0    179902
1     20098
Name: target, dtype: int64
------------------------
0    0.89951
1    0.10049
Name: target, dtype: float64


In [6]:
# thus, we now know that 89.95% people are in class 0 and the rest r in class 1
# 89.98~90, so we'll have to use feature engineerng to cover uo the minority class
# The idea: use per-value frequency as a signal to detect synthetic rows, and engineer a feature from it.

In [7]:
#Step 3: Splitting the data
# Step 3.1: dropping the target column
X=train.drop(columns=['target','ID_code'])
y=train['target']

# Step 3.2: SPlittign the data
X_train,X_test,y_train,y_test=train_test_split(X,y,train_size=0.8,test_size=0.2,random_state=23,stratify=y)


print("Training set size:", X_train.shape)
print("Validation set size:", X_test.shape)
print("Class balance in y_train:", y_train.value_counts(normalize=True))
print("Class balance in y_test:", y_test.value_counts(normalize=True))

Training set size: (160000, 200)
Validation set size: (40000, 200)
Class balance in y_train: 0    0.899513
1    0.100487
Name: target, dtype: float64
Class balance in y_test: 0    0.8995
1    0.1005
Name: target, dtype: float64


In [8]:
#Step 4: checking the recall of the given model
#Step 4.1: Scaling the data/transforming it
std_scaler=StandardScaler()
X_train_scaled=std_scaler.fit_transform(X_train)
X_test_scaled=std_scaler.transform(X_test)

In [9]:
# Step 4.2: Creatinf the model and checking the evaluation metrics of the data provided
model=LogisticRegression(random_state=23,max_iter=1000,)
model.fit(X_train_scaled,y_train)
y_test_predictions=model.predict(X_test_scaled)

In [10]:
# Step 4.3:checking recall, precision and F1 scores
recall=recall_score(y_test,y_test_predictions)
precision=precision_score(y_test,y_test_predictions)
f1=f1_score(y_test,y_test_predictions)

print("Recall score: ",recall)
print("Precision score: ",precision)
print("F1 score: ",f1)

Recall score:  0.26293532338308456
Precision score:  0.6990740740740741
F1 score:  0.3821402747650036


In [11]:
#0: default recall score : 0.2629
# Step 5: Experimenting to improve the recall

In [12]:
# Experiement 1: checking if imbalance is the issue
model_weighted=LogisticRegression(random_state=23,max_iter=1000,class_weight='balanced')
model_weighted.fit(X_train_scaled,y_train)
y_test_predictions=model_weighted.predict(X_test_scaled)

recall=recall_score(y_test,y_test_predictions)
precision=precision_score(y_test,y_test_predictions)
f1=f1_score(y_test,y_test_predictions)

print("Recall score: ",recall)
print("Precision score: ",precision)
print("F1 score: ",f1)

Recall score:  0.7728855721393035
Precision score:  0.28591147510812553
F1 score:  0.41741116410290857


In [13]:
# 1: recall score imporoved but precision dropped: 0.7728 and 0.2859 resp

In [14]:
# Experiment 2: Threshold tuning-find the approximate threshold
y_test_proba=model_weighted.predict_proba(X_test_scaled)[:,1]
print(y_test_proba[:10])
# predict_proba gives two columns: [probability of 0, probability of 1]
# we only want the second column — probability of being class 1

[0.05101944 0.71843377 0.08664771 0.49359665 0.14433549 0.83390958
 0.55867331 0.40010056 0.03036329 0.08898019]


In [15]:
# finding the threshold with nearest recall to our target recall(.88 or higher)
thresholds=[0.5,0.4,0.35,0.3,0.25,0.2]
for t in thresholds:
    y_predict_t=(y_test_proba>=t).astype(int)
#.astype(int) is use to convert the boolean value of y_test_proba into 0s or 1
    r=recall_score(y_test,y_predict_t)
    p=precision_score(y_test,y_predict_t)
    f=f1_score(y_test,y_predict_t)
    print("Recall: ",r," |Precision: ",p," |F1: ",f," for threshold: ",t)

Recall:  0.7728855721393035  |Precision:  0.28591147510812553  |F1:  0.41741116410290857  for threshold:  0.5
Recall:  0.8415422885572139  |Precision:  0.23745349898224188  |F1:  0.37039470082662723  for threshold:  0.4
Recall:  0.872636815920398  |Precision:  0.21627620221948213  |F1:  0.3466403162055336  for threshold:  0.35
Recall:  0.8995024875621891  |Precision:  0.1958511617830255  |F1:  0.32166525819508074  for threshold:  0.3
Recall:  0.922636815920398  |Precision:  0.17639225757359586  |F1:  0.29616321315926064  for threshold:  0.25
Recall:  0.9432835820895522  |Precision:  0.158953722334004  |F1:  0.27206198880757637  for threshold:  0.2


In [16]:
# manually, we can see that when threshold is 0.3, the recall is 0.89
# BUt the precision dropped to 0.19

In [17]:
#Experiment 3: Frequency encoding

# calculating frequencies using training data
X_train_freq=X_train.copy()
X_test_freq=X_test.copy()

for col in X_train.columns:
    freq_map=X_train[col].value_counts()
#     creating new features
    X_train_freq[col+'_freq']=X_train[col].map(freq_map)
    X_test_freq[col+"_freq"]=X_test[col].map(freq_map).fillna(0)
print(X_train_freq.shape)   #doubling the columns (400) — original 200 + 200 new _freq columns
X_train_freq.head()

C:\Users\MYPC~1\AppData\Local\Temp/ipykernel_32756/3707920898.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.  To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train_freq[col+'_freq']=X_train[col].map(freq_map)
C:\Users\MYPC~1\AppData\Local\Temp/ipykernel_32756/3707920898.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.  To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test_freq[col+"_freq"]=X_test[col].map(freq_map).fillna(0)


(160000, 400)


,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,var_8,var_9,...,var_190_freq,var_191_freq,var_192_freq,var_193_freq,var_194_freq,var_195_freq,var_196_freq,var_197_freq,var_198_freq,var_199_freq
180199,17.4750,-3.7118,15.4337,5.5652,13.5085,-0.9316,4.9055,19.2559,-4.1088,7.4656,...,2,3,6,2,3,1,2,7,4,2
4658,11.0297,-6.1190,9.1147,6.1303,12.5903,0.8657,5.2894,17.7909,6.2846,6.9023,...,3,3,2,2,2,4,1,10,2,1
159958,14.7526,-7.9744,11.1112,6.9413,9.0797,-9.0874,5.7142,15.0309,-4.8852,7.3363,...,2,3,7,1,4,3,1,1,4,5
129726,8.8067,-0.2506,14.0076,6.7576,12.0449,-4.9717,6.1768,15.3310,3.8691,7.3475,...,1,1,3,4,2,5,1,3,5,1
97753,11.4607,5.5549,10.0541,4.7954,11.4067,-4.5019,5.3595,18.6151,-1.0369,6.2229,...,2,3,4,2,2,6,3,4,2,1


In [18]:
# scaling the new feature set
std_scalar_freq=StandardScaler()
X_train_freq_scaled=std_scalar_freq.fit_transform(X_train_freq)
X_test_freq_scaled=std_scalar_freq.transform(X_test_freq)


In [20]:
# training a model with the new feature set
model_freq=LogisticRegression(random_state=23,max_iter=1000,class_weight='balanced')
model_freq.fit(X_train_freq_scaled,y_train)

y_test_proba_freq=model_freq.predict_proba(X_test_freq_scaled)[:,1]

# applying the best threshold in the 2nd experiment
y_test_pred_freq=(y_test_proba_freq>=0.3).astype(int)

# calculating recall,precision,F1 scor with the new frequwncy set
recall_freq=recall_score(y_test,y_test_pred_freq)
precision_freq=precision_score(y_test,y_test_pred_freq)
f1_freq=f1_score(y_test,y_test_pred_freq)

print("Evaluation mteric values with new features:")
print("Recall: ",recall_freq)
print("Precision: ",precision_freq)
print("F1: ",f1_freq)

Evaluation mteric values with new features:
Recall:  0.9945273631840796
Precision:  0.11110802323319345
F1:  0.19988500862435318


In [21]:
# after 3rd experiment, recall increased significantly and now it is 0.9945 but precision dropped to 0.11 and the F1 score dropped significantly to 0.19
# After adding new frequency features, we want to find a more accurate threshold so that precision is not compromised
threshold=0.3
step=0.01
max_iterations=100

for i in range(max_iterations):
    y_pred_t=(y_test_proba_freq>=threshold).astype(int)
    r=recall_score(y_test,y_pred_t)
    p=precision_score(y_test,y_pred_t)
    if 0.88<=r<=0.92 and 0.30<=p<=0.45:
        print("New threshold value: ",threshold)
        break
    if r>0.92 or p<0.30:
        threshold+=step
    elif r<0.88 or p>0.45:
        threshold-=step

In [24]:
# describing the data
# 4th Experiment: 
X_train_full=X_train_freq.copy()
X_test_full=X_test_freq.copy()

original_cols=X_train.columns

X_train_full['row_mean'] = X_train[original_cols].mean(axis=1)
X_train_full['row_std'] = X_train[original_cols].std(axis=1)
X_train_full['row_min'] = X_train[original_cols].min(axis=1)
X_train_full['row_max'] = X_train[original_cols].max(axis=1)
X_train_full['row_sum'] = X_train[original_cols].sum(axis=1)

X_test_full['row_mean'] = X_test[original_cols].mean(axis=1)
X_test_full['row_std'] = X_test[original_cols].std(axis=1)
X_test_full['row_min'] = X_test[original_cols].min(axis=1)
X_test_full['row_max'] = X_test[original_cols].max(axis=1)
X_test_full['row_sum'] = X_test[original_cols].sum(axis=1)

print(X_train_full.shape)  # should now be 405 columns: 200 original + 200 freq + 5 new row stats
X_train_full.head()

(160000, 405)


,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,var_8,var_9,...,var_195_freq,var_196_freq,var_197_freq,var_198_freq,var_199_freq,row_mean,row_std,row_min,row_max,row_sum
180199,17.4750,-3.7118,15.4337,5.5652,13.5085,-0.9316,4.9055,19.2559,-4.1088,7.4656,...,1,2,7,4,2,6.556392,10.166186,-47.0177,41.7262,1311.2785
4658,11.0297,-6.1190,9.1147,6.1303,12.5903,0.8657,5.2894,17.7909,6.2846,6.9023,...,4,1,10,2,1,7.588322,9.227049,-24.1618,40.3733,1517.6645
159958,14.7526,-7.9744,11.1112,6.9413,9.0797,-9.0874,5.7142,15.0309,-4.8852,7.3363,...,3,1,1,4,5,6.690457,9.549611,-22.1779,36.5401,1338.0914
129726,8.8067,-0.2506,14.0076,6.7576,12.0449,-4.9717,6.1768,15.3310,3.8691,7.3475,...,5,1,3,5,1,6.381538,9.200027,-31.5777,31.4015,1276.3077
97753,11.4607,5.5549,10.0541,4.7954,11.4067,-4.5019,5.3595,18.6151,-1.0369,6.2229,...,6,3,4,2,1,6.462856,9.233554,-41.0406,38.0620,1292.5713


In [25]:
#scaling the data
scaler_full = StandardScaler()
X_train_full_scaled = scaler_full.fit_transform(X_train_full)
X_test_full_scaled = scaler_full.transform(X_test_full)

In [27]:
# training the model like before
model_full = LogisticRegression(random_state=23,max_iter=1000,class_weight='balanced')
model_full.fit(X_train_full_scaled, y_train)
y_test_proba_full = model_full.predict_proba(X_test_full_scaled)[:, 1]

In [30]:
thresholds = [0.5, 0.45, 0.4, 0.35, 0.3, 0.25, 0.2]

for t in thresholds:
    y_pred_t = (y_test_proba_full >= t).astype(int)
    r = recall_score(y_test, y_pred_t)
    p = precision_score(y_test, y_pred_t)
    f = f1_score(y_test, y_pred_t)
    print("Recall: ",r," |Precision: ",p," |F1: ",f," for threshold: ",t)

Recall:  0.9534825870646766  |Precision:  0.1591777408637874  |F1:  0.2728113879003559  for threshold:  0.5
Recall:  0.9629353233830846  |Precision:  0.149534515393827  |F1:  0.25886916106597113  for threshold:  0.45
Recall:  0.9716417910447761  |Precision:  0.1407364704186784  |F1:  0.24586139611002708  for threshold:  0.4
Recall:  0.9788557213930348  |Precision:  0.1330245765863223  |F1:  0.23421921966608134  for threshold:  0.35
Recall:  0.9843283582089553  |Precision:  0.12654705938789215  |F1:  0.22426251806511943  for threshold:  0.3
Recall:  0.9898009950248756  |Precision:  0.12060865084416962  |F1:  0.21501715706141414  for threshold:  0.25
Recall:  0.9927860696517413  |Precision:  0.11512389303949  |F1:  0.20632253728642694  for threshold:  0.2


In [31]:
# experiment 4 results:recall further increased to 0.95 when the threshold is 0.5 and the precision is 0.15
# lowering the threshold will lower the F1 score further
# it is slightly better than 3rd experiment in terms of F1 but worse than experiment 2
# thus, adding the statictics of rows made the model 

In [33]:
# Experiment 5:removing outliers for better predictions
X_train_clipped=X_train.copy()
X_test_clipped=X_test.copy()
for col in original_cols:
    lower=X_train[col].quantile(0.01) #values at 1 percentile
    upper=X_train[col].quantile(0.99) #values at 99 percentile
    X_train_clipped[col]=X_train[col].clip(lower,upper)
    X_test_clipped[col]=X_test[col].clip(lower,upper)
print(X_train_clipped.shape)   # same columns but just tamed values
X_train_clipped.describe()  
    

(160000, 200)


,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,var_8,var_9,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
count,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,...,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000,160000.000000
mean,10.684793,-1.611573,10.716613,6.795323,11.078847,-5.062576,5.411756,16.542031,0.289238,7.570264,...,3.238552,7.441396,1.925918,3.334534,17.989853,-0.141729,2.301204,8.908654,15.860900,-3.322564
std,3.013797,4.021915,2.619819,2.025915,1.607777,7.799392,0.857477,3.392311,3.306483,1.226597,...,4.516020,2.995189,1.456105,3.943031,3.115684,1.418268,5.426525,0.913719,2.984488,10.342982
min,4.434488,-9.877000,5.469193,2.515596,7.559198,-21.070201,3.602600,9.769199,-6.742200,4.791498,...,-6.616902,1.347098,-1.245402,-6.017105,11.146100,-3.208003,-8.849005,6.816400,8.555998,-26.191801
25%,8.457900,-4.727550,8.716075,5.250350,9.884900,-11.202650,4.771400,13.937675,-2.315100,6.622300,...,-0.069050,5.161275,0.887000,0.591200,15.631875,-1.172425,-1.954075,8.254000,13.820775,-11.210550
50%,10.525650,-1.592450,10.579750,6.820550,11.103550,-4.819050,5.387250,16.457200,0.398300,7.634800,...,3.194900,7.348050,1.899600,3.401250,17.955800,-0.174550,2.395100,8.889900,15.927000,-2.816100
75%,12.761400,1.372200,12.517900,8.326825,12.257000,0.924200,6.004325,19.103200,2.945125,8.587500,...,6.413125,9.510700,2.947200,6.211100,20.388600,0.827200,6.543200,9.596000,18.057125,4.829450
max,18.164012,7.538811,17.321000,11.208604,14.484200,12.273203,7.439204,23.886801,6.648606,9.874703,...,13.520901,14.810905,5.360312,11.902904,24.464007,3.215500,13.905402,10.869603,22.138403,16.551104


In [36]:
# scaling the clipped data
scaler_clipped=StandardScaler()
X_train_clipped_scaled=scaler_clipped.fit_transform(X_train_clipped)
X_test_clipped_scaled=scaler_clipped.transform(X_test_clipped)

In [38]:
# trainig the new model without outliers
model_clipped=LogisticRegression(random_state=23,max_iter=1000,class_weight='balanced')
model_clipped.fit(X_train_clipped_scaled,y_train)
y_test_proba_clipped = model_clipped.predict_proba(X_test_clipped_scaled)[:, 1]

thresholds = [0.5, 0.45, 0.4, 0.35, 0.3, 0.25, 0.2]

for t in thresholds:
    y_pred_t = (y_test_proba_clipped >= t).astype(int)
    r = recall_score(y_test, y_pred_t)
    p = precision_score(y_test, y_pred_t)
    f = f1_score(y_test, y_pred_t)
    print("Recall: ",r," |Precision: ",p," |F1: ",f," for threshold: ",t)

Recall:  0.7738805970149254  |Precision:  0.285439031103771  |F1:  0.41705208123868887  for threshold:  0.5
Recall:  0.8097014925373134  |Precision:  0.2605042016806723  |F1:  0.3941871026339691  for threshold:  0.45
Recall:  0.8427860696517413  |Precision:  0.23753768491902125  |F1:  0.3706175135371657  for threshold:  0.4
Recall:  0.8738805970149254  |Precision:  0.21593214088143095  |F1:  0.34629602247523295  for threshold:  0.35
Recall:  0.8987562189054726  |Precision:  0.1952972972972973  |F1:  0.32087033747779753  for threshold:  0.3
Recall:  0.9221393034825871  |Precision:  0.17612124667426834  |F1:  0.2957555449178235  for threshold:  0.25
Recall:  0.9432835820895522  |Precision:  0.15902038077665018  |F1:  0.2721596210435656  for threshold:  0.2


In [42]:
#Experiment 5: results are very much similar to experiment 2 with 0.8987 recall and 0.19529 precision fro threshold=0.3

In [40]:
experiment_log = {
    0: {
        "change": "Baseline LR, default threshold",
        "recall": 0.263,
        "precision": 0.699,
        "f1": 0.382
    },
    1: {
        "change": "class_weight='balanced'",
        "recall": 0.773,
        "precision": 0.286,
        "f1": 0.417
    },
    2: {
        "change": "class_weight='balanced' + threshold=0.3",
        "recall": 0.900,
        "precision": 0.196,
        "f1": 0.322
    },
    3: {
        "change": "Frequency features (Idea A), threshold=0.3",
        "recall": 0.995,
        "precision": 0.111,
        "f1": 0.200
    },
    4: {
        "change": "Row stats (Idea B) on top of Idea A, threshold=0.5",
        "recall": 0.953,
        "precision": 0.159,
        "f1": 0.273
    },
    5: {
        "change": "Outlier clipping on original features, threshold=0.3",
        "recall": 0.899,
        "precision": 0.195,
        "f1": 0.321
    }
}
log_df = pd.DataFrame.from_dict(experiment_log, orient='index')
log_df.index.name = 'Experiment'
log_df

,change,recall,precision,f1
Experiment,,,,
0,"Baseline LR, default threshold",0.263,0.699,0.382
1,class_weight='balanced',0.773,0.286,0.417
2,class_weight='balanced' + threshold=0.3,0.900,0.196,0.322
3,"Frequency features (Idea A), threshold=0.3",0.995,0.111,0.200
4,"Row stats (Idea B) on top of Idea A, threshold...",0.953,0.159,0.273
5,"Outlier clipping on original features, thresho...",0.899,0.195,0.321
